# NLP â€” dataset audit and cleaning

Interactive inspection of the implemented AbdomenAtlas audit and cleaning stages. The notebook calls pure loading/cleaning functions and displays results inline; it does not rewrite the raw or processed datasets.

## Imports and configuration

In [1]:
import sys
sys.dont_write_bytecode = True  # no __pycache__/ or .pyc; must run before project imports

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
ROOT = next(
    (p for p in (current, *current.parents) if (p / "scripts").is_dir()),
    None
)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print({"project_root": str(ROOT), "current_working_dir": str(current)})

from scripts.nlp.audit_dataset import load_and_validate, ID, TEXT, COUNT, LOCATION, DIAMETER, ATTENUATION
from scripts.nlp.prepare_nlp_dataset import clean_report

RAW_PATH = ROOT / 'data/raw/nlp/AbdomenAtlas3.0MiniWithMeta.csv'
df = load_and_validate(RAW_PATH)
print({'rows': len(df), 'columns': len(df.columns), 'path': str(RAW_PATH)})

{'project_root': 'D:\\University\\Graduation Project\\LiverSira AI', 'current_working_dir': 'D:\\University\\Graduation Project\\LiverSira AI\\notebooks\\nlp'}
{'rows': 9262, 'columns': 37, 'path': 'D:\\University\\Graduation Project\\LiverSira AI\\data\\raw\\nlp\\AbdomenAtlas3.0MiniWithMeta.csv'}


## Input data inspection

In [2]:
overview = pd.DataFrame({
    'value': [len(df), df[ID].nunique(), int(df[ID].duplicated().sum()), int(df[ID].isna().sum()), int(df[TEXT].isna().sum()), int(df[TEXT].astype('string').str.strip().eq('').fillna(False).sum())]
}, index=['rows', 'unique_ids', 'duplicate_ids', 'missing_ids', 'missing_reports', 'empty_reports'])
display(overview)
display(df[[ID, TEXT, COUNT, LOCATION, DIAMETER, ATTENUATION]].head(3))
display(df.isna().sum().sort_values(ascending=False).head(12).rename('missing').to_frame())

,value
rows,9262
unique_ids,9262
duplicate_ids,0
missing_ids,0
missing_reports,1
empty_reports,0


,BDMAP ID,narrative report,number of liver lesion instances,largest liver lesion location (1-8),largest liver lesion diameter (cm),"largest liver lesion attenuation (hyperattenuating, isoattenuating, hypoattenuating)"
0,BDMAP_00000001,"The patient has a normal-sized spleen, measuri...",25.0,2,3.1,hypoattenuating
1,BDMAP_00000002,"The spleen is normal in size, measuring 220.1 ...",0.0,NaN,0.0,NaN
2,BDMAP_00000003,"The patient has an enlarged spleen, measuring ...",0.0,NaN,0.0,NaN


,missing
fusion structured report,9145
radiologist note,9145
fusion narrative report,9145
"largest colon lesion attenuation (hyperattenuating, isoattenuating, hypoattenuating)",9080
scanner,8777
"largest pancreatic lesion location (head, body, tail)",8205
"largest pancreatic lesion attenuation (hyperattenuating, isoattenuating, hypoattenuating)",8151
contrast,8046
sex,7888
age,7883


## Target consistency and text cleaning

In [3]:
count = pd.to_numeric(df[COUNT], errors='coerce')
target_checks = {
    'zero_lesion_rows': int(count.eq(0).sum()),
    'positive_lesion_rows': int(count.gt(0).sum()),
    'missing_counts': int(count.isna().sum()),
    'negative_counts': int(count.lt(0).sum()),
    'positive_missing_location': int((count.gt(0) & df[LOCATION].isna()).sum()),
    'positive_missing_diameter': int((count.gt(0) & pd.to_numeric(df[DIAMETER], errors='coerce').isna()).sum()),
    'positive_missing_attenuation': int((count.gt(0) & df[ATTENUATION].isna()).sum()),
}
display(pd.Series(target_checks, name='count').to_frame())
cleaned = df[TEXT].map(clean_report)
cleaning_preview = pd.DataFrame({'original': df[TEXT].head(5), 'cleaned': cleaned.head(5)})
display(cleaning_preview)
print({'usable_reports': int(cleaned.notna().sum()), 'excluded_by_cleaning': int(cleaned.isna().sum()), 'remaining_missing_after_cleaning': int(cleaned.isna().sum())})

,count
zero_lesion_rows,7792
positive_lesion_rows,1470
missing_counts,0
negative_counts,0
positive_missing_location,19
positive_missing_diameter,0
positive_missing_attenuation,0


,original,cleaned
0,"The patient has a normal-sized spleen, measuri...","The patient has a normal-sized spleen, measuri..."
1,"The spleen is normal in size, measuring 220.1 ...","The spleen is normal in size, measuring 220.1 ..."
2,"The patient has an enlarged spleen, measuring ...","The patient has an enlarged spleen, measuring ..."
3,"The spleen, liver, pancreas, and kidneys are a...","The spleen, liver, pancreas, and kidneys are a..."
4,"The spleen is normal in size, measuring 73.8 c...","The spleen is normal in size, measuring 73.8 c..."


{'usable_reports': 9261, 'excluded_by_cleaning': 1, 'remaining_missing_after_cleaning': 1}


## Inline text inspection

In [4]:
lengths = cleaned.dropna().str.len()
display(pd.Series({'mean_characters': lengths.mean(), 'median_characters': lengths.median(), 'shortest': lengths.min(), 'longest': lengths.max(), 'duplicate_clean_text_rows': int(cleaned.dropna().str.lower().duplicated(keep=False).sum())}, name='text_quality').to_frame())
sample = df.loc[cleaned.notna(), [ID, TEXT]].head(5).copy()
sample['cleaned_text'] = cleaned.loc[sample.index]
display(sample)

,text_quality
mean_characters,762.191556
median_characters,631.000000
shortest,120.000000
longest,3289.000000
duplicate_clean_text_rows,0.000000


,BDMAP ID,narrative report,cleaned_text
0,BDMAP_00000001,"The patient has a normal-sized spleen, measuri...","The patient has a normal-sized spleen, measuri..."
1,BDMAP_00000002,"The spleen is normal in size, measuring 220.1 ...","The spleen is normal in size, measuring 220.1 ..."
2,BDMAP_00000003,"The patient has an enlarged spleen, measuring ...","The patient has an enlarged spleen, measuring ..."
3,BDMAP_00000004,"The spleen, liver, pancreas, and kidneys are a...","The spleen, liver, pancreas, and kidneys are a..."
4,BDMAP_00000005,"The spleen is normal in size, measuring 73.8 c...","The spleen is normal in size, measuring 73.8 c..."


## Summary

The implemented audit identifies missingness and target relationships without deleting data. `clean_report` only normalizes whitespace and returns missing values explicitly. Persisting a cleaned dataset remains an explicit pipeline operation, not a side effect of this notebook.

In [5]:
assert set([ID, TEXT, COUNT, LOCATION, DIAMETER, ATTENUATION]).issubset(df.columns)
assert len(df) == 9262
print('Audit and cleaning inspection completed without writing files.')

Audit and cleaning inspection completed without writing files.
